# 01 — Raw Data Check

## What did we get from BigQuery?

The dataset contains US-granted AI-related patents collected from
Google Patents Public Data through BigQuery.

Before starting the analysis, we first check whether the data is really ready to use.

We look for:

- Missing values
- Duplicate records
- Inconsistent values
- Invalid dates
- Problems in nested fields
- Unusual or extreme values


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

ROOT = Path.cwd().parent

PROCESSED_DIR = ROOT / "Data" / "raw"
TABLES_DIR = ROOT / "Outputs" / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED_DIR / "Combined_data.parquet"

print("Project root:", ROOT)
print("Dataset:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

Project root: /Users/janakdobariya/Bramha/NLP_Engineering/BA/ai_patent_business_analytics
Dataset: /Users/janakdobariya/Bramha/NLP_Engineering/BA/ai_patent_business_analytics/Data/raw/Combined_data.parquet
File exists: True


In [5]:
df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (80816, 22)


In [6]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Number of rows:    {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

print("\nColumn names:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

DATASET OVERVIEW
Number of rows:    80,816
Number of columns: 22

Column names:
01. publication_number
02. application_number
03. application_number_formatted
04. family_id
05. country_code
06. kind_code
07. application_kind
08. pct_number
09. publication_date
10. filing_date
11. grant_date
12. priority_date
13. title_raw
14. abstract_raw
15. claims_raw
16. inventors_raw
17. assignees_raw
18. cpc_raw
19. citations_raw
20. raw_grant_year
21. claims_raw_length
22. citation_count_raw


In [7]:
dtype_table = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values
})

dtype_table

,column,dtype
0,publication_number,str
1,application_number,str
2,application_number_formatted,str
3,family_id,int64
4,country_code,str
5,kind_code,str
6,application_kind,str
7,pct_number,str
8,publication_date,int64
9,filing_date,float64


In [8]:
dtype_table.to_csv(
    TABLES_DIR / "01_data_types.csv",
    index=False
)

In [9]:
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (
        df.isna().mean() * 100
    ).round(2)
})

missing_table = (
    missing_table[
        missing_table["missing_count"] > 0
    ]
    .sort_values(
        "missing_count",
        ascending=False
    )
)

missing_table

,missing_count,missing_percent
pct_number,73022,90.36
assignees_raw,1208,1.49
priority_date,965,1.19
abstract_raw,728,0.90
filing_date,563,0.70
inventors_raw,484,0.60
application_number_formatted,322,0.40
title_raw,242,0.30


In [10]:
missing_table.to_csv(
    TABLES_DIR / "02_missing_values.csv"
)

In [11]:
exact_duplicates = df.duplicated().sum()

print(
    "Exact duplicate rows:",
    f"{exact_duplicates:,}"
)

Exact duplicate rows: 234


In [12]:
duplicate_publication_count = (
    df["publication_number"]
    .duplicated()
    .sum()
)

print(
    "Duplicate publication numbers:",
    f"{duplicate_publication_count:,}"
)

Duplicate publication numbers: 250


In [13]:
duplicate_patents = df[
    df.duplicated(
        subset=["publication_number"],
        keep=False
    )
].sort_values("publication_number")

duplicate_patents[
    [
        "publication_number",
        "application_number",
        "grant_date",
        "country_code",
        "kind_code"
    ]
].head(20)

,publication_number,application_number,grant_date,country_code,kind_code
128,US-10885433-B2,US-201816107717-A,20210105,US,B2
80746,US-10885433-B2,US-201816107717-A,20210105,US,B2
323,US-10891527-B2,US-201916357504-A,20210112,US,B2
80658,US-10891527-B2,US-201916357504-A,20210112,US,B2
379,US-10891949-B2,US-201816125944-A,20210112,US,B2
80695,US-10891949-B2,US-201816125944-A,20210112,US,B2
689,US-10902342-B2,US-201615267553-A,20210126,US,B2
80721,US-10902342-B2,US-201615267553-A,20210126,US,B2
979,US-10913455-B2,US-201716314595-A,20210209,US,B2
80743,US-10913455-B2,US-201716314595-A,20210209,US,B2


In [14]:
country_counts = (
    df["country_code"]
    .value_counts(dropna=False)
    .to_frame("count")
)

country_counts

,count
country_code,
US,80119
US,397
UNKNOWN,57
USA,55
us,55
US,53
U.S.,44
United States,36


In [15]:
country_counts.to_csv(
    TABLES_DIR / "03_country_code_audit.csv"
)

In [16]:
kind_counts = (
    df["kind_code"]
    .value_counts(dropna=False)
    .to_frame("count")
)

kind_counts

,count
kind_code,
B2,71922
B1,8494
b2,358
b1,42


In [17]:
for column in [
    "country_code",
    "kind_code"
]:

    values = df[column].dropna().astype(str)

    whitespace_count = (
        values != values.str.strip()
    ).sum()

    print(
        column,
        "values with leading/trailing whitespace:",
        whitespace_count
    )

country_code values with leading/trailing whitespace: 450
kind_code values with leading/trailing whitespace: 0


In [18]:
date_columns = [
    "publication_date",
    "filing_date",
    "grant_date",
    "priority_date"
]

date_audit = []

for column in date_columns:

    values = df[column]

    parsed = pd.to_datetime(
        values.astype("Int64").astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

    invalid_count = (
        values.notna() & parsed.isna()
    ).sum()

    date_audit.append({
        "column": column,
        "missing": values.isna().sum(),
        "invalid": invalid_count,
        "earliest_valid": parsed.min(),
        "latest_valid": parsed.max()
    })

date_audit = pd.DataFrame(date_audit)

date_audit

,column,missing,invalid,earliest_valid,latest_valid
0,publication_date,0,0,2021-01-05,2026-04-21
1,filing_date,563,173,1900-01-01,2050-12-31
2,grant_date,0,0,2021-01-05,2026-04-21
3,priority_date,965,421,1900-01-01,2050-12-31


In [19]:
for column in [
    "filing_date",
    "priority_date"
]:

    numeric_values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    suspicious = (
        (numeric_values < 19000101) |
        (numeric_values > 20261231)
    )

    print(
        column,
        "suspicious values:",
        suspicious.sum()
    )

filing_date suspicious values: 135
priority_date suspicious values: 381


In [20]:
json_columns = [
    "title_raw",
    "abstract_raw",
    "claims_raw",
    "inventors_raw",
    "assignees_raw",
    "cpc_raw",
    "citations_raw"
]

In [21]:
def json_status(value):

    if pd.isna(value):
        return "missing"

    try:
        obj = json.loads(value)

        if isinstance(obj, list) and len(obj) == 0:
            return "empty"

        return "valid"

    except (json.JSONDecodeError, TypeError):
        return "invalid"


json_audit_results = []

for column in json_columns:

    status = df[column].apply(json_status)

    counts = status.value_counts()

    json_audit_results.append({
        "column": column,
        "valid": counts.get("valid", 0),
        "empty": counts.get("empty", 0),
        "missing": counts.get("missing", 0),
        "invalid": counts.get("invalid", 0)
    })

json_audit = pd.DataFrame(json_audit_results)

json_audit

,column,valid,empty,missing,invalid
0,title_raw,80574,0,242,0
1,abstract_raw,80088,0,728,0
2,claims_raw,80816,0,0,0
3,inventors_raw,80010,322,484,0
4,assignees_raw,79188,320,1208,100
5,cpc_raw,80816,0,0,0
6,citations_raw,80752,64,0,0


In [22]:
family_counts = (
    df["family_id"]
    .value_counts()
)

repeated_families = family_counts[
    family_counts > 1
]

print(
    "Unique family IDs:",
    f"{df['family_id'].nunique():,}"
)

print(
    "Family IDs occurring more than once:",
    f"{len(repeated_families):,}"
)

print(
    "Rows belonging to repeated families:",
    f"{df['family_id'].isin(repeated_families.index).sum():,}"
)

Unique family IDs: 63,817
Family IDs occurring more than once: 10,723
Rows belonging to repeated families: 27,722


In [23]:
numeric_columns = [
    "claims_raw_length",
    "citation_count_raw"
]

df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
claims_raw_length,80816.0,10886.073141,24586.901163,610.0,7142.75,9202.5,11870.0,2698347.0
citation_count_raw,80816.0,54.357021,202.377828,0.0,11.00,20.0,39.0,20832.0


In [24]:
outlier_results = []

for column in numeric_columns:

    series = df[column].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (series < lower_bound) |
        (series > upper_bound)
    )

    outlier_results.append({
        "column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "potential_outliers": outliers.sum(),
        "outlier_percent": round(
            outliers.mean() * 100,
            2
        )
    })

outlier_table = pd.DataFrame(
    outlier_results
)

outlier_table

,column,Q1,Q3,IQR,lower_bound,upper_bound,potential_outliers,outlier_percent
0,claims_raw_length,7142.75,11870.0,4727.25,51.875,18960.875,3507,4.34
1,citation_count_raw,11.00,39.0,28.00,-31.000,81.000,9017,11.16


In [25]:
audit_summary = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Total columns",
        "Unique publication numbers",
        "Duplicate publication numbers",
        "Columns containing missing values",
        "Invalid JSON fields detected"
    ],

    "Value": [
        len(df),
        df.shape[1],
        df["publication_number"].nunique(),
        df["publication_number"].duplicated().sum(),
        (df.isna().sum() > 0).sum(),
        json_audit["invalid"].sum()
    ]
})

audit_summary

,Metric,Value
0,Total rows,80816
1,Total columns,22
2,Unique publication numbers,80566
3,Duplicate publication numbers,250
4,Columns containing missing values,8
5,Invalid JSON fields detected,100


## Preliminary Audit Conclusion

The initial audit identified several data-quality issues requiring preprocessing:

1. Missing values are present across several variables.
2. Duplicate patent records exist.
3. Categorical variables contain inconsistent formatting and labels.
4. Some date values are invalid or outside plausible ranges.
5. Some JSON records are malformed or empty.
6. Numerical variables contain extreme observations requiring investigation.
7. Repeated patent-family identifiers exist but should not automatically be considered duplicates.

These issues will be addressed systematically in the preprocessing stage.